In [1]:
# Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/PlantVillage'

In [3]:
# Extract the dataset
import zipfile

zip_path = "/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/plant_data.zip"
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/data")

In [4]:
# List the class folders (should be 15)
from pathlib import Path

base_dir = Path("/content/data/PlantVillage")
print(sorted([p.name for p in base_dir.iterdir() if p.is_dir()]))  # should list 15 class names

['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [6]:
# Random 80/10/10 split per class, saved to splits.json
import random, json

random.seed(42)
split = {"train": [], "val": [], "test": []}

for class_folder in sorted(base_dir.iterdir()):
    if class_folder.is_dir():
        images = sorted(class_folder.glob("*"))
        random.shuffle(images)

        n = len(images)
        train_end = int(0.8 * n)
        val_end = int(0.9 * n)  # 0.8 + 0.1

        split["train"] += [str(p.relative_to(base_dir)) for p in images[:train_end]]
        split["val"] += [str(p.relative_to(base_dir)) for p in images[train_end:val_end]]
        split["test"] += [str(p.relative_to(base_dir)) for p in images[val_end:]]

manifest_path = Path("/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/splits.json")
manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path, "w") as f:
    json.dump(split, f)

print(
    f"{len(split['train'])} train / {len(split['val'])} val / {len(split['test'])} test images saved to manifest"
)

16505 train / 2064 val / 2070 test images saved to manifest


In [7]:
# Copy the files into the train/val/test folders
import shutil

train_dir = Path("/content/data/train")
val_dir = Path("/content/data/val")
test_dir = Path("/content/data/test")

for split_name, target_dir in [("train", train_dir), ("val", val_dir), ("test", test_dir)]:
    for rel_path in split[split_name]:
        src = base_dir / rel_path
        dst = target_dir / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(str(src), str(dst))

print("Train/val/test folders built.")

Train/val/test folders built.


In [8]:
# Image counts per class and split
for c in sorted(train_dir.iterdir()):
    if c.is_dir():
        n_train = len(list(c.glob("*")))
        n_val = len(list((val_dir / c.name).glob("*")))
        n_test = len(list((test_dir / c.name).glob("*")))
        print(f"{c.name}: {n_train} train, {n_val} val, {n_test} test")

Pepper__bell___Bacterial_spot: 797 train, 100 val, 100 test
Pepper__bell___healthy: 1182 train, 148 val, 148 test
Potato___Early_blight: 800 train, 100 val, 100 test
Potato___Late_blight: 800 train, 100 val, 100 test
Potato___healthy: 121 train, 15 val, 16 test
Tomato_Bacterial_spot: 1701 train, 213 val, 213 test
Tomato_Early_blight: 800 train, 100 val, 100 test
Tomato_Late_blight: 1527 train, 191 val, 191 test
Tomato_Leaf_Mold: 761 train, 95 val, 96 test
Tomato_Septoria_leaf_spot: 1416 train, 177 val, 178 test
Tomato_Spider_mites_Two_spotted_spider_mite: 1340 train, 168 val, 168 test
Tomato__Target_Spot: 1123 train, 140 val, 141 test
Tomato__Tomato_YellowLeaf__Curl_Virus: 2567 train, 321 val, 321 test
Tomato__Tomato_mosaic_virus: 298 train, 37 val, 38 test
Tomato_healthy: 1272 train, 159 val, 160 test
